In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import benchutils as bu

plt.style.use('bmh')

# the interpreter running this notebook, so workers land in the same env
PYTHON_PATH = sys.executable
WORKER_SCRIPT = 'tmp/symscan_ncpu_worker.py'
N_SEQUENCE = 1_000_000
MAX_DISTANCE = 2
DISTANCE_TYPE = 'levenshtein'
N_REPS = 1

# Sweep 1..N threads over whatever cores this process was pinned to by
# bench_pinned.sh (one logical CPU per physical core, one core class).
MAX_NCPU = bu.available_cpus()
ncpus = np.arange(1, MAX_NCPU + 1, 1)
print(f'scaling over {MAX_NCPU} cores: {bu.affinity_list()}')

scaling over 8 cores: [0, 2, 4, 6, 8, 10, 12, 14]


In [2]:
bu.describe_env()

{'colab': False,
 'platform': 'Linux-6.8.0-136-generic-x86_64-with-glibc2.39',
 'python': '3.12.13',
 'git_sha': 'f5112ac',
 'cpu_model': '13th Gen Intel(R) Core(TM) i9-13900',
 'n_cpus_total': 32,
 'affinity': [0, 2, 4, 6, 8, 10, 12, 14],
 'n_cpus_visible': 8,
 'governor': 'performance',
 'no_turbo': '1',
 'gpu': {'name': 'NVIDIA GeForce RTX 4090',
  'memory_total': '24564 MiB',
  'clocks_max_sm': '3105 MHz',
  'clocks_applications_gr': '[N/A]'},
 'thread_env': {'RAYON_NUM_THREADS': '8',
  'OMP_NUM_THREADS': '8',
  'MKL_NUM_THREADS': '8',
  'OPENBLAS_NUM_THREADS': '8',
  'NUMEXPR_NUM_THREADS': '8',
  'NUMBA_NUM_THREADS': '8',
  'OMP_PROC_BIND': 'close',
  'OMP_PLACES': 'cores'},
 'packages': {'symscan': '0.8.3',
  'pyrepseq': '1.5.2',
  'pybktree': '1.1',
  'rapidfuzz': '3.14.5',
  'pwseqdist': '0.6',
  'numba': '0.66.0',
  'numpy': '2.4.6',
  'scipy': '1.18.0',
  'pandas': '3.0.5'},
 'timeout_seconds': 100}

In [3]:
!mkdir -p tmp

In [4]:
%%writefile tmp/symscan_ncpu_worker.py
import sys
import time
import pandas as pd

import symscan


def main():
    n_sequence, max_distance, distance_type = sys.argv[1:5]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    N_FILES=1
    seqs = []
    for i in range(1,N_FILES+1):
        seqs += pd.read_csv(f'../data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()
    seqs = seqs[:n_sequence]

    t0 = time.perf_counter()
    symscan.get_neighbors_within(seqs, max_distance=max_distance, distance_type=distance_type)
    print(time.perf_counter() - t0)


if __name__ == '__main__':
    main()

Overwriting tmp/symscan_ncpu_worker.py


In [5]:
def measure_runtime_seconds(n_cpu, n_sequence=N_SEQUENCE, max_distance=MAX_DISTANCE, distance_type=DISTANCE_TYPE):
    cmd = [PYTHON_PATH, WORKER_SCRIPT, str(n_sequence), str(max_distance), distance_type]
    env = os.environ | {'RAYON_NUM_THREADS': str(n_cpu)}
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return float(result.stdout)


In [6]:
rows = []
for rep in range(N_REPS):
    for ncpu in ncpus:
        runtime_s = measure_runtime_seconds(ncpu)
        rows.append({'algorithm': 'symscan', 'n_cpu': int(ncpu), 'n_sequence': N_SEQUENCE,
                      'distance': MAX_DISTANCE, 'measure': DISTANCE_TYPE,
                      'runtime_s': runtime_s})
        print(ncpu, rep, runtime_s)

ncpu_df = pd.DataFrame(rows)
ncpu_df.to_csv('../data/symscan_ncpu_benchmark.csv')

1 0 95.34689167997567


2 0 52.31676618399797


3 0 36.53174369398039


4 0 29.824117793003097


5 0 24.445036509016063


6 0 21.345103005995043


7 0 19.382882767997216


8 0 17.75998490600614
